# 03b · GSE65391 · RNA_array · probes to genes

Reads `probes_corrected.rds` (notebook 03), `GPL10558.soft.gz` and NCBI gene_info. Writes
`data/run_artifacts/GSE65391/expression.rds`.

**Why one value per gene.** The two RNA sequencing studies measure genes, not probes, and the projection
and preservation steps (notebooks 05b, 15, 15b, 26, 26b) need one value per gene in every study. Probes of
the same gene also correlate for technical reasons, because they measure the same transcript, and would
form tight groups in WGCNA without adding biology.

**How.** Only probes rated Perfect or Good (notebook 02) with an Entrez gene identifier are used. Where
several probes measure one gene, the probe with the highest mean expression is kept: the "MaxMean" method of
`collapseRows()`, which gave the most consistent results between data sets for network analysis (Miller JA,
Cai C, Langfelder P, et al. Strategies for aggregating gene expression data: the collapseRows R function.
*BMC Bioinformatics* 2011;12:322).

Genes are named with their current NCBI symbol, so all three studies use the same names. A gene is
**expressed** when the probe kept for it is detected (notebook 02: detection p-value below 0.05 in at least
10% of arrays).

## Load the corrected probes

In [1]:
if (!requireNamespace("WGCNA", quietly = TRUE)) install.packages("WGCNA", repos = "https://cloud.r-project.org")
suppressMessages({library(WGCNA); library(GEOquery)})
source("../src/paths.R")
p  <- readRDS(art("GSE65391", "probes_corrected.rds"))
gi <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
use <- p$quality_ok & !is.na(p$entrez)
c(probes = nrow(p$E), quality_ok_with_entrez = sum(use), arrays = ncol(p$E))

probes quality_ok_with_entrez                 arrays 
                 47231                  32754                    972

**Explanation**
- WGCNA is installed if missing and loaded (see notebook 04); it provides `collapseRows()`.
- `probes_corrected.rds` (notebook 03) holds the corrected values `E` and, for each probe, `detected`,
  `quality_ok` and `entrez`.
- `read.delim()` reads the NCBI gene table (see notebook 00 for the file). `quote = ""` treats quote marks as
  ordinary characters; `colClasses = "character"` reads every column as text.
- `use` is TRUE for probes rated Perfect or Good that have an Entrez identifier.

## Keep one probe per gene

**Result.** 32,754 of the 47,231 probes are rated Perfect or Good and have an Entrez identifier.


**Result.** 32,754 of the 47,231 probes are rated Perfect or Good and have an Entrez identifier.

In [2]:
col <- collapseRows(p$E[use, ], rowGroup = p$entrez[use], rowID = rownames(p$E)[use], method = "MaxMean")
E <- col$datETcollapsed
chosen <- col$group2row[match(rownames(E), col$group2row[, "group"]), "selectedRowID"]
c(genes = nrow(E))

genes 
23058

**Explanation**
- `collapseRows(values, rowGroup, rowID, method = "MaxMean")` groups the probes by gene (`rowGroup`, the
  Entrez identifier) and keeps, for each gene, the probe with the highest mean over all arrays. For example,
  a gene with three probes of mean 7.1, 9.4 and 6.8 keeps the second one.
- `col$datETcollapsed` is the result: one row per gene, named by Entrez identifier.
- `col$group2row` lists, for each gene, the probe kept. `chosen` holds that probe for each row of `E`.

## Name the genes and mark the expressed ones

In [3]:
gpl <- Table(suppressMessages(getGEO(filename = raw("GSE65391", "GPL10558.soft.gz"))))
symbol <- gi$Symbol[match(rownames(E), gi$GeneID)]
old    <- gpl$Symbol[match(rownames(E), gpl$Entrez_Gene_ID)]
symbol[is.na(symbol)] <- old[is.na(symbol)]         # identifiers retired since 2010 keep the array symbol
keep <- !duplicated(symbol)
E <- E[keep, ]; chosen <- chosen[keep]; rownames(E) <- symbol[keep]
expressed <- setNames(p$detected[chosen], rownames(E))
c(genes = nrow(E), expressed = sum(expressed))

genes expressed 
    23040     13259

**Explanation**
- `symbol` is each gene's current symbol in the NCBI table, found by its Entrez identifier.
- `old` is the symbol in the array annotation, used only where NCBI no longer lists the identifier.
- `duplicated()` is TRUE for the second and later copies of a symbol; `keep` keeps the first.
- `expressed` is TRUE when the probe kept for a gene is detected. `setNames(..., rownames(E))` names each
  value by gene. For example, `expressed["IFI27"]` is TRUE if IFI27's probe is detected in at least 10% of
  arrays.

## Save

**Result.** 23,040 genes, of which 13,259 are expressed.


**Result.** 23,040 genes, of which 13,259 are expressed.

In [4]:
saveRDS(list(E = E, expressed = expressed, meta = p$meta), art("GSE65391", "expression.rds"))

**Explanation**
- `expression.rds` holds `E` (genes by samples), `expressed` and `meta`, the sample table. Every later array
  notebook reads it.